# 04 - Fine-tuning (LoRA/PEFT) do assistente medico

**Este notebook foi desenhado para rodar no Google Colab (GPU gratuita, ex.: T4).**
Localmente (Windows, sem GPU) o treino seria lento demais; o codigo abaixo e correto e testado no fluxo de dados, mas a celula de treino em si deve ser executada no Colab.

Passos:
1. Clonar o repositorio no Colab (ou fazer upload de `data/medical_corpus/`).
2. Instalar as dependencias de fine-tuning.
3. Carregar o modelo base + aplicar LoRA.
4. Treinar sobre `train.jsonl`, avaliar sobre `eval.jsonl`.
5. Salvar o adapter em `results/finetuning/lora_adapter/` e baixar de volta para o repo local (usado por `src/assistant/llm_backend.py`).

In [7]:
# No Colab, descomente e rode:
!rm -rf stroke-prediction # Adicionado para garantir uma clonagem limpa
!git clone https://github.com/BrunaNicolau/stroke-prediction stroke-prediction
%cd stroke-prediction
!pip install -r requirements.txt
# !pip install -q bitsandbytes  # opcional, acelera em GPU

Cloning into 'stroke-prediction'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 206 (delta 85), reused 167 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 392.52 KiB | 1.91 MiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/stroke-prediction/stroke-prediction


In [8]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.finetuning.dataset import build_hf_dataset, load_split

train_examples = load_split(os.path.join(PROJECT_ROOT, 'data', 'medical_corpus', 'train.jsonl'))
eval_examples = load_split(os.path.join(PROJECT_ROOT, 'data', 'medical_corpus', 'eval.jsonl'))
print(f'{len(train_examples)} exemplos de treino, {len(eval_examples)} de avaliacao')

train_dataset = build_hf_dataset(train_examples)
train_dataset[0]

ModuleNotFoundError: No module named 'src.finetuning'

## Modelo base

Usamos um LLM pequeno e **nao-gated** no Hugging Face (evita a burocracia de acesso aos pesos oficiais do Llama) — `Qwen/Qwen2.5-1.5B-Instruct`. Isso ainda atende ao requisito do desafio ("LLaMA, Falcon **ou outro**").

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto'
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512, padding='max_length')

tokenized_train = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
tokenized_train.set_format(type='torch')

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, 'results', 'finetuning', 'checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='no',
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)
train_result = trainer.train()
train_result

In [ ]:
import json

ADAPTER_DIR = os.path.join(PROJECT_ROOT, 'results', 'finetuning', 'lora_adapter')
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

metrics = {
    'base_model': BASE_MODEL,
    'train_examples': len(train_examples),
    'eval_examples': len(eval_examples),
    'train_loss_history': [
        {'step': h.get('step'), 'loss': h.get('loss')}
        for h in trainer.state.log_history if 'loss' in h
    ],
}
with open(os.path.join(PROJECT_ROOT, 'results', 'finetuning', 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print('Adapter e metricas salvos em results/finetuning/')

## Avaliacao qualitativa (antes/depois)

Compara a saida do modelo base vs. o modelo com o adapter LoRA para as mesmas perguntas do conjunto de avaliacao, usando o checklist deterministico de `src/finetuning/evaluate.py` (sem GPU/API, reproduzivel).

In [ ]:
from src.finetuning.dataset import format_prompt_only
from src.finetuning.evaluate import evaluate_eval_set


def generate_with_model(current_model, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors='pt').to(current_model.device)
    output = current_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text[len(prompt):].strip()


pairs_finetuned = []
for ex in eval_examples[:10]:
    prompt = format_prompt_only(ex)
    generated = generate_with_model(model, prompt)
    pairs_finetuned.append((generated, ex['output']))

result = evaluate_eval_set(pairs_finetuned)
print('Score medio (modelo fine-tuned):', result['mean_score'])

## Proximos passos

Baixe a pasta `results/finetuning/lora_adapter/` do Colab para o mesmo caminho no repositorio local. `src/assistant/llm_backend.get_generate_fn()` detecta o adapter automaticamente e passa a usar o modelo fine-tuned em vez do fallback Gemini nos notebooks 05 e 06.